# FNEWS scratch SVC experiment

This notebook runs the independent, standard-library-only synthetic experiment. It does not train or alter the existing FNEWS model. The optional TSV route requires an explicit source-label mapping check: `1 = SOURCE_TRUE`, `0 = SOURCE_NON_TRUE`; neither is a fact-check verdict.

In [1]:
from pathlib import Path
import sys
root = Path.cwd()
while not (root / 'experiments.py').exists():
    root = root.parent
sys.path.insert(0, str(root))
from experiments import fnews_fakeddit_tsv_experiment, run_synthetic
from src.metrics import classification_report


In [2]:
results = run_synthetic()
for name in ('primal_linear', 'dual_linear', 'xor_linear', 'xor_rbf'):
    result = results[name]
    print(name, 'accuracy=', round(result['report']['accuracy'], 3), 'macro_f1=', round(result['report']['macro_f1'], 3))
    print('diagnostics:', {key: value for key, value in result['diagnostics'].items() if key not in {'objective_history', 'history'}})
print('sensitivity:', results['sensitivity'])


primal_linear accuracy= 1.0 macro_f1= 1.0
diagnostics: {'optimizer': 'primal', 'epochs': 289, 'stopping_reason': 'objective_change_heuristic', 'margin_violations': 0, 'objective_change_heuristic_triggered': True}
dual_linear accuracy= 1.0 macro_f1= 1.0
diagnostics: {'optimizer': 'smo', 'iterations': 12, 'support_vectors': 2, 'equality_residual': 3.469446951953614e-18, 'kkt_violations': 0, 'max_kkt_residual': 0.0, 'max_box_constraint_violation': 3.469446951953614e-18, 'alpha_updates': 11, 'stopping_reason': 'kkt_satisfied_for_max_passes', 'converged': True}
xor_linear accuracy= 0.5 macro_f1= 0.333
diagnostics: {'optimizer': 'smo', 'iterations': 12, 'support_vectors': 4, 'equality_residual': 0.0, 'kkt_violations': 0, 'max_kkt_residual': 0.0, 'max_box_constraint_violation': 0.0, 'alpha_updates': 6, 'stopping_reason': 'kkt_satisfied_for_max_passes', 'converged': True}
xor_rbf accuracy= 1.0 macro_f1= 1.0
diagnostics: {'optimizer': 'smo', 'iterations': 12, 'support_vectors': 4, 'equality_res

## Bounded Fakeddit source-taxonomy experiment

This is a read-only educational experiment on the local official validation TSV. It validates the local six-way relationship before sampling and treats the held-out subset as a fixed demonstration partition, not a model-selection benchmark.

In [3]:
tsv = root.parent / 'data/raw/fakeddit/multimodal_only_samples/multimodal_validate.tsv'
fnews_result = fnews_fakeddit_tsv_experiment(tsv)
print('taxonomy verified:', fnews_result['taxonomy_verified'], fnews_result['mapping'])
print('source rows:', fnews_result['source_rows_by_label'])
print('sample/train/test:', fnews_result['sampled_rows'], fnews_result['train_rows'], fnews_result['test_rows'])
print('accuracy/macro-F1/balanced accuracy:', fnews_result['report']['accuracy'], fnews_result['report']['macro_f1'], fnews_result['report']['balanced_accuracy'])
print('confusion matrix [0=SOURCE_NON_TRUE, 1=SOURCE_TRUE]:', fnews_result['report']['confusion_matrix'])
print('diagnostics:', {key: value for key, value in fnews_result['diagnostics'].items() if key not in {'objective_history', 'history'}})


taxonomy verified: True {'1': 'SOURCE_TRUE', '0': 'SOURCE_NON_TRUE'}
source rows: {0: 36022, 1: 23320}
sample/train/test: 600 475 125
accuracy/macro-F1/balanced accuracy: 0.616 0.6147919876733436 0.6164874551971327
confusion matrix [0=SOURCE_NON_TRUE, 1=SOURCE_TRUE]: [[42, 20], [28, 35]]
diagnostics: {'optimizer': 'primal', 'epochs': 164, 'stopping_reason': 'objective_change_heuristic', 'margin_violations': 364, 'objective_change_heuristic_triggered': True}


## Interpretation

The linear model should not perfectly separate standard XOR, whereas a sufficiently localized RBF kernel can. These are optimization demonstrations, not fake-news accuracy claims. For a real text run, prepare a bounded TSV with `text` and `label` columns and call `optional_tsv_experiment` only after verifying its label semantics.